In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

print("Pandas version:", pd.__version__)

Pandas version: 3.0.3


In [5]:
PROJECT_DIR = Path.cwd().parent

GTFS_DIR = PROJECT_DIR / "extracted"

print("Project directory:", PROJECT_DIR)
print("GTFS directory:", GTFS_DIR)

print("Directory exists:", GTFS_DIR.exists())

Project directory: C:\Users\sairaj\BI_BD\transit_realtime_collector
GTFS directory: C:\Users\sairaj\BI_BD\transit_realtime_collector\extracted
Directory exists: True


In [6]:
stops = pd.read_csv(
    GTFS_DIR / "stops.txt",
    dtype={
        "stop_id": "string",
        "parent_station": "string",
        "location_type": "Int64",
        "platform_code": "string"
    }
)

print("Stops loaded successfully!")
print("Shape:", stops.shape)

display(stops.head())

Stops loaded successfully!
Shape: (15742, 7)


,stop_name,parent_station,stop_id,stop_lat,stop_lon,location_type,platform_code
0,Aachen Hbf,328974,567808,50.7678,6.0915,<NA>,<NA>
1,Aachen Schanz,518166,191170,50.7699,6.0738,<NA>,<NA>
2,Aachen Süd(Gr),<NA>,181154,50.7319,6.0454,1,<NA>
3,Aachen Süd(Gr),181154,413906,50.7319,6.0454,<NA>,<NA>
4,Aachen West,<NA>,512476,50.7804,6.0707,1,<NA>


In [7]:
print("Number of stops:", len(stops))
print("Number of columns:", len(stops.columns))

print("\nColumns:")
print(stops.columns.tolist())

Number of stops: 15742
Number of columns: 7

Columns:
['stop_name', 'parent_station', 'stop_id', 'stop_lat', 'stop_lon', 'location_type', 'platform_code']


In [8]:
stops.info()

<class 'pandas.DataFrame'>
RangeIndex: 15742 entries, 0 to 15741
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   stop_name       15742 non-null  str    
 1   parent_station  8307 non-null   string 
 2   stop_id         15742 non-null  string 
 3   stop_lat        15742 non-null  float64
 4   stop_lon        15742 non-null  float64
 5   location_type   7435 non-null   Int64  
 6   platform_code   933 non-null    string 
dtypes: Int64(1), float64(2), str(1), string(3)
memory usage: 1.2 MB


In [9]:
missing = pd.DataFrame({
    "missing_count": stops.isnull().sum(),
    "missing_percentage": stops.isnull().mean() * 100
})

display(missing.sort_values("missing_percentage", ascending=False))

,missing_count,missing_percentage
platform_code,14809,94.0732
location_type,8307,52.7697
parent_station,7435,47.2303
stop_id,0,0.0000
stop_name,0,0.0000
stop_lon,0,0.0000
stop_lat,0,0.0000


In [10]:
print("Duplicate stop IDs:", stops["stop_id"].duplicated().sum())

print(
    "Duplicate complete rows:",
    stops.duplicated().sum()
)

Duplicate stop IDs: 0
Duplicate complete rows: 0


In [11]:
display(
    stops["location_type"]
    .value_counts(dropna=False)
    .sort_index()
)

location_type
1       7435
<NA>    8307
Name: count, dtype: Int64

In [12]:
display(
    stops[["stop_lat", "stop_lon"]].describe()
)

,stop_lat,stop_lon
count,15742.0000,15742.0000
mean,50.5021,10.4573
std,1.7574,2.4410
min,41.9108,2.3549
25%,49.0971,8.4760
50%,50.6321,10.1265
75%,51.7084,12.4776
max,54.9334,19.1331


In [13]:
print("Missing latitude:", stops["stop_lat"].isna().sum())
print("Missing longitude:", stops["stop_lon"].isna().sum())

Missing latitude: 0
Missing longitude: 0


In [14]:
station_ids = set(
    stops.loc[
        stops["location_type"] == 1,
        "stop_id"
    ]
)

child_stops = stops[stops["parent_station"].notna()].copy()

child_stops["parent_exists"] = (
    child_stops["parent_station"].isin(station_ids)
)

print("Child stops:", len(child_stops))

print(
    "Parent station exists:",
    child_stops["parent_exists"].sum()
)

print(
    "Invalid parent references:",
    (~child_stops["parent_exists"]).sum()
)

Child stops: 8307
Parent station exists: 8307
Invalid parent references: 0


In [15]:
children_per_station = (
    child_stops
    .groupby("parent_station")
    .size()
    .reset_index(name="number_of_child_stops")
    .sort_values("number_of_child_stops", ascending=False)
)

display(children_per_station.head(15))


,parent_station,number_of_child_stops
6970,682344,13
3311,367315,11
1693,236738,10
1024,185854,9
2784,32395,8
6457,638161,8
5062,518443,8
6996,68549,8
968,181609,7
4170,442985,7


In [16]:
station_lookup = (
    stops.loc[
        stops["location_type"] == 1,
        ["stop_id", "stop_name"]
    ]
    .rename(columns={
        "stop_id": "parent_station",
        "stop_name": "station_name"
    })
)

child_with_station = child_stops.merge(
    station_lookup,
    on="parent_station",
    how="left"
)

display(
    child_with_station[
        [
            "stop_name",
            "stop_id",
            "parent_station",
            "station_name",
            "platform_code"
        ]
    ].head(20)
)

,stop_name,stop_id,parent_station,station_name,platform_code
0,Aachen Hbf,567808,328974,"Aachen, Hbf",<NA>
1,Aachen Schanz,191170,518166,"Aachen, Schanz (Jakobstraße)",<NA>
2,Aachen Süd(Gr),413906,181154,Aachen Süd(Gr),<NA>
3,Aachen West,613962,512476,Aachen West,<NA>
4,Aachen-Rothe Erde,2108,521028,Aachen-Rothe Erde,<NA>
5,Aalen Hbf,644501,211777,"Aalen, Hauptbahnhof",<NA>
6,Aarau,77500,141000,Aarau,<NA>
7,Abenden,269588,600951,Abenden,<NA>
8,Abensberg,53458,208342,Abensberg,<NA>
9,Achern,383467,316312,Achern Bahnhof,<NA>


## Findings – Stops

1. The GTFS dataset contains 15,742 stop records.
2. 7,435 records are explicitly classified as stations.
3. 8,307 records are classified as stops/platforms by the
   GTFS default for missing location_type.
4. All 8,307 child stops have valid parent station references.
5. No duplicate stop IDs were detected.
6. No missing latitude or longitude values were found.
7. Parent station names and child stop names can differ.
8. Station-level analysis should use the parent-child hierarchy
   rather than relying only on stop names.

# Part 2 – Stop Times Exploration

## Objective

Understand the scheduled arrival and departure times of trains
at each stop, and investigate how stops are ordered within a trip.

In [17]:
stop_times = pd.read_csv(
    GTFS_DIR / "stop_times.txt",
    dtype={
        "trip_id": "string",
        "stop_id": "string",
        "arrival_time": "string",
        "departure_time": "string",
        "stop_sequence": "Int64",
        "stop_headsign": "string",
        "pickup_type": "Int64",
        "drop_off_type": "Int64"
    }
)

print("Stop times loaded successfully!")
print("Shape:", stop_times.shape)

display(stop_times.head(10))

Stop times loaded successfully!
Shape: (1473009, 8)


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type
0,1003776,05:00:00,05:00:00,377944,0,Augsburg Hbf,<NA>,<NA>
1,1003776,05:03:00,05:04:00,201550,1,Augsburg Hbf,<NA>,<NA>
2,1003776,05:06:00,05:07:00,100515,2,Augsburg Hbf,<NA>,<NA>
3,1003776,05:13:00,05:13:00,500163,3,Augsburg Hbf,<NA>,<NA>
4,1003776,05:22:00,05:23:00,160607,4,Augsburg Hbf,<NA>,<NA>
5,1003776,05:26:00,05:27:00,151203,5,Augsburg Hbf,<NA>,<NA>
6,1003776,05:35:00,05:44:00,413601,6,Augsburg Hbf,<NA>,<NA>
7,1003776,05:50:00,05:51:00,566277,7,Augsburg Hbf,<NA>,<NA>
8,1003776,05:54:00,05:56:00,103370,8,Augsburg Hbf,<NA>,<NA>
9,1003776,05:58:00,05:59:00,264476,9,Augsburg Hbf,<NA>,<NA>


In [18]:
print("Rows:", len(stop_times))
print("Columns:", len(stop_times.columns))

print("\nColumn names:")
print(stop_times.columns.tolist())

Rows: 1473009
Columns: 8

Column names:
['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type']


In [19]:
stop_times.info()

<class 'pandas.DataFrame'>
RangeIndex: 1473009 entries, 0 to 1473008
Data columns (total 8 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   trip_id         1473009 non-null  string
 1   arrival_time    1473009 non-null  string
 2   departure_time  1473009 non-null  string
 3   stop_id         1473009 non-null  string
 4   stop_sequence   1473009 non-null  Int64 
 5   stop_headsign   1473009 non-null  string
 6   pickup_type     5471 non-null     Int64 
 7   drop_off_type   5231 non-null     Int64 
dtypes: Int64(3), string(5)
memory usage: 153.9 MB


In [20]:
display(
    stop_times.describe(include="all").T
)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
trip_id,1473009,100912,1055394,99,NaN,NaN,NaN,NaN,NaN,NaN,NaN
arrival_time,1473009,15658,07:18:00,1255,NaN,NaN,NaN,NaN,NaN,NaN,NaN
departure_time,1473009,14855,07:45:00,1168,NaN,NaN,NaN,NaN,NaN,NaN,NaN
stop_id,1473009,8307,619423,4227,NaN,NaN,NaN,NaN,NaN,NaN,NaN
stop_sequence,1473009.0000,<NA>,<NA>,<NA>,9.7070,8.6484,0.0000,3.0000,8.0000,14.0000,98.0000
stop_headsign,1473009,1852,Ringbahn S 42,22942,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pickup_type,5471.0000,<NA>,<NA>,<NA>,1.0046,0.0674,1.0000,1.0000,1.0000,1.0000,2.0000
drop_off_type,5231.0000,<NA>,<NA>,<NA>,1.0096,0.1379,1.0000,1.0000,1.0000,1.0000,3.0000


In [21]:
stop_times_missing = pd.DataFrame({
    "missing_count": stop_times.isna().sum(),
    "missing_percentage": stop_times.isna().mean() * 100
})

display(
    stop_times_missing.sort_values(
        "missing_percentage",
        ascending=False
    )
)

,missing_count,missing_percentage
drop_off_type,1467778,99.6449
pickup_type,1467538,99.6286
trip_id,0,0.0000
arrival_time,0,0.0000
stop_id,0,0.0000
departure_time,0,0.0000
stop_headsign,0,0.0000
stop_sequence,0,0.0000


In [22]:
print(
    "Duplicate complete rows:",
    stop_times.duplicated().sum()
)

print(
    "Unique trip IDs:",
    stop_times["trip_id"].nunique()
)

print(
    "Unique stop IDs:",
    stop_times["stop_id"].nunique()
)

print(
    "Unique trip-stop combinations:",
    stop_times[["trip_id", "stop_sequence"]]
    .drop_duplicates()
    .shape[0]
)

Duplicate complete rows: 0
Unique trip IDs: 100912
Unique stop IDs: 8307
Unique trip-stop combinations: 1473009


In [23]:
display(
    stop_times["stop_sequence"]
    .describe()
)

count   1473009.0000
mean          9.7070
std           8.6484
min           0.0000
25%           3.0000
50%           8.0000
75%          14.0000
max          98.0000
Name: stop_sequence, dtype: Float64

In [24]:
# Select one trip for investigation
sample_trip_id = stop_times["trip_id"].iloc[0]

sample_trip = (
    stop_times[
        stop_times["trip_id"] == sample_trip_id
    ]
    .sort_values("stop_sequence")
)

display(sample_trip)

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type
0,1003776,05:00:00,05:00:00,377944,0,Augsburg Hbf,<NA>,<NA>
1,1003776,05:03:00,05:04:00,201550,1,Augsburg Hbf,<NA>,<NA>
2,1003776,05:06:00,05:07:00,100515,2,Augsburg Hbf,<NA>,<NA>
3,1003776,05:13:00,05:13:00,500163,3,Augsburg Hbf,<NA>,<NA>
4,1003776,05:22:00,05:23:00,160607,4,Augsburg Hbf,<NA>,<NA>
5,1003776,05:26:00,05:27:00,151203,5,Augsburg Hbf,<NA>,<NA>
6,1003776,05:35:00,05:44:00,413601,6,Augsburg Hbf,<NA>,<NA>
7,1003776,05:50:00,05:51:00,566277,7,Augsburg Hbf,<NA>,<NA>
8,1003776,05:54:00,05:56:00,103370,8,Augsburg Hbf,<NA>,<NA>
9,1003776,05:58:00,05:59:00,264476,9,Augsburg Hbf,<NA>,<NA>


In [25]:
sample_trip_details = sample_trip.merge(
    stops[
        ["stop_id", "stop_name", "parent_station"]
    ],
    on="stop_id",
    how="left"
)

display(
    sample_trip_details[
        [
            "trip_id",
            "stop_sequence",
            "stop_id",
            "stop_name",
            "arrival_time",
            "departure_time",
            "parent_station"
        ]
    ]
)

,trip_id,stop_sequence,stop_id,stop_name,arrival_time,departure_time,parent_station
0,1003776,0,377944,Schongau,05:00:00,05:00:00,255760
1,1003776,1,201550,Peiting Nord,05:03:00,05:04:00,207389
2,1003776,2,100515,Peiting Ost,05:06:00,05:07:00,417881
3,1003776,3,500163,Hohenpeißenberg,05:13:00,05:13:00,230515
4,1003776,4,160607,Peißenberg,05:22:00,05:23:00,624900
5,1003776,5,151203,Peißenberg Nord,05:26:00,05:27:00,353566
6,1003776,6,413601,Weilheim(Oberbay),05:35:00,05:44:00,252632
7,1003776,7,566277,Raisting,05:50:00,05:51:00,321944
8,1003776,8,103370,Dießen,05:54:00,05:56:00,617974
9,1003776,9,264476,St Alban,05:58:00,05:59:00,676208


In [26]:
known_stop_ids = set(stops["stop_id"])

missing_stop_ids = (
    set(stop_times["stop_id"]) - known_stop_ids
)

print("Unique stop IDs in stop_times:", stop_times["stop_id"].nunique())
print("Unmatched stop IDs:", len(missing_stop_ids))

Unique stop IDs in stop_times: 8307
Unmatched stop IDs: 0


In [27]:
trips = pd.read_csv(
    GTFS_DIR / "trips.txt",
    dtype={
        "route_id": "string",
        "service_id": "string",
        "trip_id": "string"
    }
)

known_trip_ids = set(trips["trip_id"])

missing_trip_ids = (
    set(stop_times["trip_id"]) - known_trip_ids
)

print("Unique trip IDs in stop_times:", stop_times["trip_id"].nunique())
print("Unmatched trip IDs:", len(missing_trip_ids))

Unique trip IDs in stop_times: 100912
Unmatched trip IDs: 0


In [28]:
stops_per_trip = (
    stop_times
    .groupby("trip_id")
    .size()
    .reset_index(name="number_of_stops")
)

display(stops_per_trip.describe())

,number_of_stops
count,100912.0000
mean,14.5970
std,9.2148
min,2.0000
25%,8.0000
50%,12.0000
75%,20.0000
max,99.0000


plt.figure(figsize=(10, 5))

plt.hist(
    stops_per_trip["number_of_stops"],
    bins=30
)

plt.title("Distribution of Stop Counts per Trip")
plt.xlabel("Number of Stop Events")
plt.ylabel("Number of Trips")

plt.grid(True)
plt.tight_layout()
plt.show()

In [29]:
child_stop_ids = set(
    stops.loc[
        stops["parent_station"].notna(),
        "stop_id"
    ]
)

timetable_stop_ids = set(stop_times["stop_id"])

print("Child stops:", len(child_stop_ids))
print("Timetable stop IDs:", len(timetable_stop_ids))

print(
    "Child stops not used in timetable:",
    len(child_stop_ids - timetable_stop_ids)
)

print(
    "Timetable stop IDs not in child stops:",
    len(timetable_stop_ids - child_stop_ids)
)

print(
    "Are both sets identical?",
    child_stop_ids == timetable_stop_ids
)

Child stops: 8307
Timetable stop IDs: 8307
Child stops not used in timetable: 0
Timetable stop IDs not in child stops: 0
Are both sets identical? True


In [30]:
def gtfs_time_to_seconds(value):
    if pd.isna(value):
        return np.nan

    hours, minutes, seconds = map(int, value.split(":"))

    return hours * 3600 + minutes * 60 + seconds


stop_times["arrival_seconds"] = (
    stop_times["arrival_time"].apply(gtfs_time_to_seconds)
)

stop_times["departure_seconds"] = (
    stop_times["departure_time"].apply(gtfs_time_to_seconds)
)

display(
    stop_times[
        [
            "trip_id",
            "stop_id",
            "arrival_time",
            "arrival_seconds",
            "departure_time",
            "departure_seconds"
        ]
    ].head(10)
)

,trip_id,stop_id,arrival_time,arrival_seconds,departure_time,departure_seconds
0,1003776,377944,05:00:00,18000,05:00:00,18000
1,1003776,201550,05:03:00,18180,05:04:00,18240
2,1003776,100515,05:06:00,18360,05:07:00,18420
3,1003776,500163,05:13:00,18780,05:13:00,18780
4,1003776,160607,05:22:00,19320,05:23:00,19380
5,1003776,151203,05:26:00,19560,05:27:00,19620
6,1003776,413601,05:35:00,20100,05:44:00,20640
7,1003776,566277,05:50:00,21000,05:51:00,21060
8,1003776,103370,05:54:00,21240,05:56:00,21360
9,1003776,264476,05:58:00,21480,05:59:00,21540


In [31]:
print(
    "Earliest arrival:",
    stop_times["arrival_time"].min()
)

print(
    "Latest arrival:",
    stop_times["arrival_time"].max()
)

print(
    "Earliest departure:",
    stop_times["departure_time"].min()
)

print(
    "Latest departure:",
    stop_times["departure_time"].max()
)

Earliest arrival: 00:00:00
Latest arrival: 36:45:00
Earliest departure: 00:00:00
Latest departure: 36:45:00


In [32]:
after_midnight = stop_times[
    stop_times["arrival_seconds"] >= 86400
]

print(
    "Arrival records at or after 24:00:",
    len(after_midnight)
)

print(
    "Percentage:",
    len(after_midnight) / len(stop_times) * 100
)

display(
    after_midnight[
        ["trip_id", "stop_id", "arrival_time"]
    ].head(10)
)

Arrival records at or after 24:00: 73238
Percentage: 4.9719994921959065


,trip_id,stop_id,arrival_time
392,1276612,392657,24:02:00
393,1276612,560809,24:08:00
394,1276612,339967,24:18:00
395,1276612,608641,24:22:00
396,1276612,264476,24:24:00
397,1276612,103370,24:27:00
398,1276612,566277,24:33:00
399,1276612,413601,24:42:00
940,1575399,392657,24:02:00
941,1575399,560809,24:08:00


In [33]:
# Count departure records at or after 24:00
after_midnight_departures = (
    stop_times["departure_seconds"] >= 24 * 60 * 60
)

print(
    "Departure records at or after 24:00:",
    after_midnight_departures.sum()
)

print(
    "Percentage:",
    round(after_midnight_departures.mean() * 100, 2),
    "%"
)

Departure records at or after 24:00: 74060
Percentage: 5.03 %


In [34]:
after_midnight_rows = stop_times[
    (stop_times["arrival_seconds"] >= 86400) |
    (stop_times["departure_seconds"] >= 86400)
]

print(
    "Distinct trips with after-midnight events:",
    after_midnight_rows["trip_id"].nunique()
)

print(
    "Stop-time records belonging to these trips:",
    len(after_midnight_rows)
)

Distinct trips with after-midnight events: 6913
Stop-time records belonging to these trips: 74060


In [35]:
trips = pd.read_csv(
    GTFS_DIR / "trips.txt",
    dtype={
        "trip_id": "string",
        "route_id": "string",
        "service_id": "string"
    }
)

routes = pd.read_csv(
    GTFS_DIR / "routes.txt",
    dtype={
        "route_id": "string",
        "agency_id": "string",
        "route_short_name": "string",
        "route_long_name": "string"
    }
)

print("Trips:", trips.shape)
print("Routes:", routes.shape)

display(trips.head())
display(routes.head())

Trips: (100912, 3)
Routes: (945, 7)


,route_id,service_id,trip_id
0,1,1666,1606876
1,1,1666,308007
2,1,2243,64352
3,1,2243,893898
4,1,3114,1003776


,route_long_name,route_short_name,agency_id,route_type,route_id,route_color,route_text_color
0,<NA>,1,43,2,856,NaN,NaN
1,<NA>,10,113,2,739,NaN,NaN
2,<NA>,12N,104,2,370,NaN,NaN
3,<NA>,12N,113,2,597,NaN,NaN
4,<NA>,12N,219,2,110,NaN,NaN


In [36]:
print("Route type distribution:")
display(
    routes["route_type"]
    .value_counts(dropna=False)
    .sort_index()
)

print("Missing route names:")
print(
    routes[
        ["route_short_name", "route_long_name"]
    ].isna().sum()
)

print("Duplicate route IDs:")
print(routes["route_id"].duplicated().sum())

Route type distribution:


route_type
2    945
Name: count, dtype: int64

Missing route names:
route_short_name      0
route_long_name     945
dtype: int64
Duplicate route IDs:
0


In [37]:
scheduled = stop_times.merge(
    trips,
    on="trip_id",
    how="left",
    validate="many_to_one"
)

scheduled = scheduled.merge(
    routes,
    on="route_id",
    how="left",
    validate="many_to_one"
)

scheduled = scheduled.merge(
    stops,
    on="stop_id",
    how="left",
    validate="many_to_one"
)

print("Scheduled timetable shape:", scheduled.shape)

display(
    scheduled[
        [
            "trip_id",
            "route_id",
            "route_short_name",
            "route_type",
            "stop_id",
            "stop_name",
            "arrival_time",
            "departure_time",
            "stop_sequence",
            "service_id"
        ]
    ].head(10)
)

Scheduled timetable shape: (1473009, 24)


,trip_id,route_id,route_short_name,route_type,stop_id,stop_name,arrival_time,departure_time,stop_sequence,service_id
0,1003776,1,RB67,2,377944,Schongau,05:00:00,05:00:00,0,3114
1,1003776,1,RB67,2,201550,Peiting Nord,05:03:00,05:04:00,1,3114
2,1003776,1,RB67,2,100515,Peiting Ost,05:06:00,05:07:00,2,3114
3,1003776,1,RB67,2,500163,Hohenpeißenberg,05:13:00,05:13:00,3,3114
4,1003776,1,RB67,2,160607,Peißenberg,05:22:00,05:23:00,4,3114
5,1003776,1,RB67,2,151203,Peißenberg Nord,05:26:00,05:27:00,5,3114
6,1003776,1,RB67,2,413601,Weilheim(Oberbay),05:35:00,05:44:00,6,3114
7,1003776,1,RB67,2,566277,Raisting,05:50:00,05:51:00,7,3114
8,1003776,1,RB67,2,103370,Dießen,05:54:00,05:56:00,8,3114
9,1003776,1,RB67,2,264476,St Alban,05:58:00,05:59:00,9,3114


In [38]:
print("Scheduled timetable shape:", scheduled.shape)

print("Missing trip IDs:", scheduled["route_id"].isna().sum())
print("Missing route IDs:", scheduled["route_short_name"].isna().sum())
print("Missing stop names:", scheduled["stop_name"].isna().sum())

Scheduled timetable shape: (1473009, 24)
Missing trip IDs: 0
Missing route IDs: 0
Missing stop names: 0


In [39]:
display(
    scheduled[
        [
            "trip_id",
            "route_id",
            "route_short_name",
            "route_type",
            "stop_id",
            "stop_name",
            "arrival_time",
            "departure_time",
            "stop_sequence",
            "service_id"
        ]
    ].head(10)
)

,trip_id,route_id,route_short_name,route_type,stop_id,stop_name,arrival_time,departure_time,stop_sequence,service_id
0,1003776,1,RB67,2,377944,Schongau,05:00:00,05:00:00,0,3114
1,1003776,1,RB67,2,201550,Peiting Nord,05:03:00,05:04:00,1,3114
2,1003776,1,RB67,2,100515,Peiting Ost,05:06:00,05:07:00,2,3114
3,1003776,1,RB67,2,500163,Hohenpeißenberg,05:13:00,05:13:00,3,3114
4,1003776,1,RB67,2,160607,Peißenberg,05:22:00,05:23:00,4,3114
5,1003776,1,RB67,2,151203,Peißenberg Nord,05:26:00,05:27:00,5,3114
6,1003776,1,RB67,2,413601,Weilheim(Oberbay),05:35:00,05:44:00,6,3114
7,1003776,1,RB67,2,566277,Raisting,05:50:00,05:51:00,7,3114
8,1003776,1,RB67,2,103370,Dießen,05:54:00,05:56:00,8,3114
9,1003776,1,RB67,2,264476,St Alban,05:58:00,05:59:00,9,3114
